In [8]:
import torch 
import sentencepiece as spm
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ensp = spm.SentencePieceProcessor(model_file="en_model.model")
hisp = spm.SentencePieceProcessor(model_file="hi_model.model")

src_vocab_size = 2079
tgt_vocab_size = 2079
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1


loadingmodel = Badassatron(src_vocab_size, tgt_vocab_size, d_model, num_heads, d_ff, num_layers, max_seq_length, dropout )

loadingmodel.load_state_dict(torch.load("tranlator_en_hi.pth", map_location=device))

# 4. CRITICAL: Switch the model to evaluation mode
# This turns off dropout layers so the model behaves deterministically
loadingmodel.eval()

def translate(model , src_sentence, ensp, hisp, max_len=300 , device=None):
    model.eval()
    if device is None:
        device = next(model.parameters()).device


    #first tokenization
    en_sp = ensp.encode(src_sentence, out_type=int)
    
    en_tensor = torch.tensor(en_sp, device=device).unsqueeze(0)
    #initialize decoder targey seq with just [start] toke
    start_id = hisp.piece_to_id("[start]")
    end_id = hisp.piece_to_id("[end]")
    with torch.no_grad():
        # --- encoder runs once ---
        src_mask = (en_tensor != 0).unsqueeze(1).unsqueeze(2)
        enc_output = model.postion_enc(model.encoder_emb(en_tensor))
        for enc_layer in model.encoder_layer:
            enc_output = enc_layer(enc_output, src_mask)
 
        # --- decoder: one new token per step, caches carried forward ---
        num_layers = len(model.decoder_layer)
        self_kv_caches = [None] * num_layers
        cross_kv_caches = [None] * num_layers
 
        tgt_token = [start_id]
 
        for step in range(max_len):
            cur_input = torch.tensor([[tgt_token[-1]]], device=device)  # only newest token
            tgt_emb = model.postion_enc(model.decoder_emb(cur_input), start_pos=step)
 
            dec_output = tgt_emb
            for i, dec_layer in enumerate(model.decoder_layer):
                dec_output, self_kv_caches[i], cross_kv_caches[i] = dec_layer(
                    dec_output,
                    enc_output,
                    src_mask,
                    None,  # no causal mask needed, single query vs. cached keys
                    self_past_kv=self_kv_caches[i],
                    cross_kv=cross_kv_caches[i],
                )
 
            logits = model.fc(dec_output[0, -1, :])
 
            # keep your repetition penalty logic
            for token_id in set(tgt_token):
                logits[token_id] /= 1.2
 
            next_token_id = logits.argmax().item()
            tgt_token.append(next_token_id)
 
            if next_token_id == end_id:
                break
 
    clean_tokens = [t for t in tgt_token if t not in (start_id, end_id)]
    hindi_translation = hisp.decode(clean_tokens)
    return hindi_translation

    # tgt_token = [start_id] "'now we don;t need when we already implement kv cache"'
    #auto regressive loop for generate text  token by token
    # with torch.no_grad():
    #     for _ in range(max_len):
    #         # tgt_tensor = torch.tensor(tgt_token, device=device).unsqueeze(0)
    #         # Create a casual mask for the decoder so it can't peer into its own future predictions
    #         # sz = tgt_tensor.size(1)
    #         # tgt_mask = (torch.triu(torch.ones(sz, sz, device=device))== 1).transpose(0,1)
    #         # tgt_mask = tgt_mask.float().masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))
    #         # cur_tar_len = tgt_tensor.size(1)
    #         # tgt_mask = torch.nn.Transformer.generate_square_subsequent_mask(cur_tar_len)


    #         #fpass
    #         output = model(
    #             src= en_tensor,
    #             tgt = tgt_tensor
    #         )
    #         logits = output[0, -1, :]

    #         # Apply a repetition penalty to tokens already generated
    #         for token_id in set(tgt_token):
    #             logits[token_id] /= 1.2  # Penalize repeating tokens

    #         next_token_id = logits.argmax().item()
    #         #get the highest scored token 
    #         next_token_id =  output[0, -1, :].argmax().item()

    #         tgt_token.append(next_token_id)

    #         if next_token_id == end_id:
    #             break
    #     clean_tokens = [t for t in tgt_token if t not in (start_id, end_id)]
    #     hindi_translation = hisp.decode(clean_tokens)
    #     return hindi_translation



test_sentences = [
    "Hello, how are you doing today?",
    "What is your favorite hobby?",
    "I am learning how to code in Python.",
    "The weather is very nice outside.",
    "Let us go for a walk in the park.",
    "Where do you live?",
    "I need to buy some groceries later.",
    "Time flies when you are having fun.",
    "She enjoys listening to music.",
    "They are planning a trip for the weekend.",
    "What time does the train arrive?",
    "Can you help me find my keys?",
    "Why is the sky blue?",
    "How do I fix this software bug?",
    "Where is the nearest coffee shop?",
    "Is this seat taken?",
    "Do you know how to swim?",
    "Which book are you reading right now?",
    "How much does this laptop cost?",
    "What is the meaning of this word?",
    "Cats are very independent animals.",
    "Earth revolves around the sun.",
    "Water boils at one hundred degrees Celsius.",
    "The capital of France is Paris.",
    "Honey never spoils.",
    "Elephants are the largest land mammals.",
    "Regular exercise is good for your health.",
    "Coding requires a lot of patience.",
    "Trees produce oxygen for us to breathe.",
    "Reading expands your knowledge.",
    "I am tired.",
    "Look at that.",
    "Keep it up.",
    "Never give up.",
    "That sounds great.",
    "See you tomorrow.",
    "I forgot my password.",
    "It is raining outside.",
    "Stop talking so loud.",
    "Trust your instincts.",
    "Please send me the updated report.",
    "We have a meeting at ten in the morning.",
    "I will reply to your email shortly.",
    "The project deadline is next Friday.",
    "Can we reschedule our call?",
    "Thank you for your feedback.",
    "He works as a software engineer.",
    "She gave an excellent presentation today.",
    "We need to find a better solution.",
    "Let us collaborate on this task.",
    "I think this is a great idea.",
    "She feels happy about her new job.",
    "I am not sure about this decision.",
    "This food tastes absolutely delicious.",
    "He was disappointed with the results.",
    "I love watching movies on weekends.",
    "It feels good to relax after work.",
    "They seem very excited about the news.",
    "I prefer tea over coffee.",
    "This song makes me feel nostalgic.",
    "Please close the door behind you.",
    "Turn left at the next intersection.",
    "Do not touch the wet paint.",
    "Read the instructions carefully before starting.",
    "Open your textbook to page twenty.",
    "Mix the ingredients thoroughly.",
    "Press the button to start the machine.",
    "Keep your room clean and organized.",
    "Drive safely on your way home.",
    "Fill out this form completely.",
    "The system is not responding.",
    "Please restart your computer.",
    "I am experiencing a slow internet connection.",
    "The application crashed unexpectedly.",
    "Your password must include a special character.",
    "Update your app to the latest version.",
    "The server is currently undergoing maintenance.",
    "This file format is not supported.",
    "Check your network settings and try again.",
    "The battery level is extremely low.",
    "I wake up at six every morning.",
    "He brushes his teeth twice a day.",
    "She takes the bus to school.",
    "They cook dinner together every night.",
    "I usually drink a glass of water first thing.",
    "He goes to the gym on Mondays.",
    "She loves to read a chapter before sleeping.",
    "We clean the house every Saturday.",
    "I check my calendar every morning.",
    "He does his homework right after school.",
    "If it rains, the game will be canceled.",
    "I wish I could travel the entire world.",
    "Technology has completely changed how we live.",
    "Balancing work and life can be difficult.",
    "She acts as if nothing happened.",
    "Although it was difficult, they finished the race.",
    "Economics is a fascinating subject to study.",
    "Innovation drives progress in every industry.",
    "He speaks three languages fluently.",
    "True success requires continuous effort."
]
start = time.perf_counter()


print("--- BadaSSatron ---")
loadingmodel.eval()
with torch.no_grad():

    for sentence in test_sentences:
        translation = translate(loadingmodel, sentence, ensp, hisp)
        print(f"English: {sentence}")
        print(f"Hindi Target Output: {translation}")
        print("-" * 30)
end =time.perf_counter()
print("total time it takes", end-start)

#without kv cache it takes total time it takes 27.00758525800029

#next step to progress by adding large data , 2nd build more models from scratch this is for use and Mrd for job


--- BadaSSatron ---
English: Hello, how are you doing today?
Hindi Target Output: तुम यहे हो?
------------------------------
English: What is your favorite hobby?
Hindi Target Output: तुम यहे हो?
------------------------------
English: I am learning how to code in Python.
Hindi Target Output: वहाँ न,ों कोें हैं।
------------------------------
English: The weather is very nice outside.
Hindi Target Output: वहाँ न, न, है।
------------------------------
English: Let us go for a walk in the park.
Hindi Target Output: वहाँ न, न,ों को है।
------------------------------
English: Where do you live?
Hindi Target Output: तुम यहे हो?
------------------------------
English: I need to buy some groceries later.
Hindi Target Output: वहाँ न,ों कोें हैं।
------------------------------
English: Time flies when you are having fun.
Hindi Target Output: तुम यहे अपनी को भी है।
------------------------------
English: She enjoys listening to music.
Hindi Target Output: वहाँ न,ों कोें हैं।
--------------------

In [7]:
# -----------------------------------------transformer with kv cache 
import math
import torch
import torch.nn as nn


class Multiheadattention(nn.Module):
    def __init__(self, dmodel, num_heads):
        super().__init__()
        assert dmodel % num_heads == 0

        self.dmodel = dmodel
        self.num_heads = num_heads
        self.d_k = dmodel // num_heads

        self.w_q = nn.Linear(dmodel, dmodel)
        self.w_k = nn.Linear(dmodel, dmodel)
        self.w_v = nn.Linear(dmodel, dmodel)
        self.w_o = nn.Linear(dmodel, dmodel)

    def scaler_product_attenion(self, Q, K, V, mask=None):
        atten_score = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            atten_score = atten_score.masked_fill(mask == 0, -1e9)
        atten_prob = torch.softmax(atten_score, dim=-1)
        output = torch.matmul(atten_prob, V)
        return output

    def splits_head(self, x):
        batch_size, seq_len, d_model = x.size()
        return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

    def combine_head(self, x):
        batch_size, _, seq_len, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.dmodel)

    def forward(self, Q, K, V, mask=None, past_kv=None, kv_override=None):
        """
        past_kv:    optional (K_cached, V_cached) tensors of shape
                    (batch, num_heads, past_len, d_k). New K/V computed from
                    K, V args are concatenated onto these (self-attention case).
        kv_override: optional (K, V) tensors to use directly instead of
                    projecting K, V args at all (cross-attention case, where
                    encoder K/V are computed once and reused every step).

        Returns (output, present_kv) where present_kv = (K_used, V_used),
        which the caller can feed back in as past_kv / kv_override on the
        next step.
        """
        Qh = self.splits_head(self.w_q(Q))

        if kv_override is not None:
            Kh, Vh = kv_override
        else:
            Kh = self.splits_head(self.w_k(K))
            Vh = self.splits_head(self.w_v(V))
            if past_kv is not None:
                past_k, past_v = past_kv
                Kh = torch.cat([past_k, Kh], dim=2)
                Vh = torch.cat([past_v, Vh], dim=2)

        atten_output = self.scaler_product_attenion(Qh, Kh, Vh, mask)
        output = self.w_o(self.combine_head(atten_output))
        present_kv = (Kh, Vh)
        return output, present_kv


class PositionWiseFF(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.ff1 = nn.Linear(d_model, d_ff)
        self.ff2 = nn.Linear(d_ff, d_model)
        self.relu = nn.GELU()

    def forward(self, x):
        return self.ff2(self.relu(self.ff1(x)))


class PositionEncoding(nn.Module):
    def __init__(self, d_model, max_seqlen):
        super(PositionEncoding, self).__init__()

        self.dropout = nn.Dropout(0.2)
        pe = torch.zeros(max_seqlen, d_model)

        postion = torch.arange(0, max_seqlen, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(postion * div_term)
        pe[:, 1::2] = torch.cos(postion * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x, start_pos=0):
        """
        start_pos lets us position-encode a single incoming token correctly
        when it's not the start of the sequence — this matters once you're
        feeding the model one cached token at a time instead of the whole
        sequence at once.
        """
        seq_len = x.size(1)
        x = x + self.pe[:, start_pos:start_pos + seq_len]
        return self.dropout(x)


class Encoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(Encoder, self).__init__()
        self.attention = Multiheadattention(dmodel=d_model, num_heads=num_heads)
        self.ff = PositionWiseFF(d_model=d_model, d_ff=d_ff)
        self.dropout = nn.Dropout(dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask):
        atten_out, _ = self.attention(x, x, x, mask)
        x = self.norm1(x + self.dropout(atten_out))
        ffoutput = self.ff(x)
        out = self.norm2(x + self.dropout(ffoutput))
        return out


class Decoder(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, droupout):
        super(Decoder, self).__init__()
        self.atten = Multiheadattention(dmodel=d_model, num_heads=num_heads)
        self.crossatten = Multiheadattention(dmodel=d_model, num_heads=num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ff = PositionWiseFF(d_model=d_model, d_ff=d_ff)
        self.dropout = nn.Dropout(droupout)

    def forward(self, x, encoder_out, src_mask, tgt_mask, self_past_kv=None, cross_kv=None):
        """
        self_past_kv: cached self-attention K/V from previous decoding steps
                      (None on the very first step / during full training pass).
        cross_kv:     cached cross-attention K/V (computed once from
                      encoder_out). Pass None the first time this layer sees
                      encoder_out; pass the returned cross_present_kv back in
                      on every subsequent step so the encoder K/V projections
                      aren't recomputed each time.
        """
        atten_out, self_present_kv = self.atten(x, x, x, tgt_mask, past_kv=self_past_kv)
        x = self.norm1(x + self.dropout(atten_out))

        atten2, cross_present_kv = self.crossatten(
            x, encoder_out, encoder_out, src_mask, kv_override=cross_kv
        )
        x = self.norm2(x + self.dropout(atten2))

        ff_x = self.ff(x)
        output = self.norm3(x + self.dropout(ff_x))
        return output, self_present_kv, cross_present_kv


class Badassatron(nn.Module):
    def __init__(self, src_vocab_size, src_tgt_size, dmodel, numheads, dff, numlayers, max_lenseq, dropout):
        super().__init__()
        self.encoder_emb = nn.Embedding(src_vocab_size, dmodel)
        self.decoder_emb = nn.Embedding(src_tgt_size, dmodel)
        self.postion_enc = PositionEncoding(d_model=dmodel, max_seqlen=max_lenseq)

        self.encoder_layer = nn.ModuleList([Encoder(dmodel, numheads, dff, dropout) for _ in range(numlayers)])
        self.decoder_layer = nn.ModuleList([Decoder(dmodel, numheads, dff, dropout) for _ in range(numlayers)])

        self.fc = nn.Linear(dmodel, src_tgt_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_len = tgt.size(1)
        nopeak = (1 - torch.triu(torch.ones(1, seq_len, seq_len), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        """Standard teacher-forced training forward pass. No caching — the
        whole target sequence is available up front, so there's nothing to
        cache across steps."""
        srcmask, tgtmask = self.generate_mask(src, tgt)
        src_emb = self.dropout(self.postion_enc(self.encoder_emb(src)))
        tgt_emb = self.dropout(self.postion_enc(self.decoder_emb(tgt)))

        enc_output = src_emb
        for enc_layer in self.encoder_layer:
            enc_output = enc_layer(enc_output, srcmask)

        dec_output = tgt_emb
        for dec_layer in self.decoder_layer:
            dec_output, _, _ = dec_layer(dec_output, enc_output, srcmask, tgtmask)
        output = self.fc(dec_output)

        return output

    @torch.no_grad()
    def generate(self, src, start_token_id, end_token_id, max_len=50):
        """
        Incremental (autoregressive) decoding with KV caching.

        At each step we only run the newest token through the decoder — its
        query attends against cached K/V from all previous steps (self-
        attention) and against the encoder's K/V, computed once and reused
        every step (cross-attention). This avoids recomputing attention over
        the whole growing sequence at every step, which is what makes cached
        generation fast.
        """
        self.eval()
        device = src.device
        batch_size = src.size(0)

        # --- Encoder runs once, no caching needed (not autoregressive) ---
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        enc_output = self.dropout(self.postion_enc(self.encoder_emb(src)))
        for enc_layer in self.encoder_layer:
            enc_output = enc_layer(enc_output, src_mask)

        # --- Decoder: one token at a time, cache grows each step ---
        tgt = torch.full((batch_size, 1), start_token_id, dtype=torch.long, device=device)
        num_layers = len(self.decoder_layer)
        self_kv_caches = [None] * num_layers   # grows every step
        cross_kv_caches = [None] * num_layers  # computed once, reused every step

        finished = torch.zeros(batch_size, dtype=torch.bool, device=device)

        for step in range(max_len):
            cur_input = tgt[:, -1:]  # only the newest token needs a forward pass
            tgt_emb = self.dropout(
                self.postion_enc(self.decoder_emb(cur_input), start_pos=step)
            )

            dec_output = tgt_emb
            for i, dec_layer in enumerate(self.decoder_layer):
                dec_output, self_kv_caches[i], cross_kv_caches[i] = dec_layer(
                    dec_output,
                    enc_output,
                    src_mask,
                    None,  # no causal mask needed: a single query position
                           # can only attend to past+current keys anyway
                    self_past_kv=self_kv_caches[i],
                    cross_kv=cross_kv_caches[i],
                )

            logits = self.fc(dec_output[:, -1, :])
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
            tgt = torch.cat([tgt, next_token], dim=1)

            finished = finished | (next_token.squeeze(-1) == end_token_id)
            if finished.all():
                break

        return tgt